# 模組 5：進階例外處理 — raise 與自定義例外

目標：學習主動拋出例外、例外鏈接，以及建立自定義例外類別。

---

## ⚾ 投手 vs 捕手

| 角色 | 語法 | 職責 |
|---|---|---|
| **投手（進攻方）** | `raise` | 程式狀態不對時，主動拋出錯誤 |
| **捕手（防守方）** | `try/except` | 捕捉並處理飛過來的錯誤 |

---
## 1. 基礎拋出 — 丟出內建例外

In [ ]:
def calculate_bmi(weight, height):
    if not isinstance(weight, (int, float)) or not isinstance(height, (int, float)):
        raise TypeError("體重和身高必須是數字！")
    if height <= 0:
        raise ValueError("身高必須大於 0")
    if weight <= 0:
        raise ValueError("體重必須大於 0")
    return weight / ((height / 100) ** 2)

try:
    result = calculate_bmi(70, -5)   # 試試：(70, 175) / ("重", 175)
    print(f"BMI = {result:.2f}")
except (TypeError, ValueError) as e:
    print(f"[輸入錯誤] {e}")

---
## 2. 無參數拋出 (Bare Raise) — 攔截後原封不動往上丟

In [ ]:
def read_config(filepath):
    try:
        with open(filepath, "r") as f:
            return f.read()
    except FileNotFoundError:
        print("【系統日誌】警告！找不到設定檔。")
        raise   # 原封不動再拋出 FileNotFoundError

try:
    read_config("config.json")
except FileNotFoundError as e:
    print(f"[上層捕捉到] {e}")

---
## 3. 例外鏈接 — `raise ... from ...`

In [ ]:
class UserNotFoundError(Exception):
    pass

def get_user_data(user_id):
    database = {"001": "Alice", "002": "Bob"}
    try:
        return database[user_id]
    except KeyError as raw_error:
        raise UserNotFoundError(f"找不到 ID 為 {user_id} 的使用者") from raw_error

try:
    get_user_data("999")
except UserNotFoundError as e:
    print(f"[UserNotFoundError] {e}")
    print(f"原始原因：{e.__cause__}")

---
## 4. 範例 A：銀行系統繼承樹（2 層）

```
Exception
└── BankError                    ← 專案根節點
    ├── InsufficientFundsError   ← 餘額不足
    └── InvalidAccountError      ← 帳號格式錯誤
```

In [ ]:
class BankError(Exception): pass
class InsufficientFundsError(BankError): pass
class InvalidAccountError(BankError): pass

# 確認繼承關係（MRO = Method Resolution Order）
print(InsufficientFundsError.__mro__)

# isinstance 也受繼承影響
err = InsufficientFundsError("餘額不足")
print(isinstance(err, BankError))    # True
print(isinstance(err, Exception))   # True

---
## 5. 範例 B：電商系統繼承樹（3 層）

```
Exception
└── ShopSystemError                    ← 專案根節點
    ├── OutOfStockError                ← 商品缺貨
    ├── UserAuthError                  ← 使用者驗證錯誤
    └── PaymentError                   ← 付款相關（中間分類層）
        ├── CreditCardExpiredError     ← 信用卡過期
        └── InsufficientBalanceError   ← 餘額不足
```

### except = elif：由上往下，先搶先贏

多個 `except` 的運作邏輯與 `if...elif...` **幾乎一模一樣**：  
**黃金法則：由小到大，由精準到寬鬆**（子類別排在父類別前面）

| except 層 | 比喻 | 捕捉範圍 |
|---|---|---|
| `InsufficientBalanceError` | 🐟 小漁網 | 只抓餘額不足 |
| `CreditCardExpiredError` | 🐟 小漁網 | 只抓卡片過期 |
| `PaymentError` | 🦈 中漁網 | 所有付款問題 |
| `ShopSystemError` | 🐳 大漁網 | 所有系統錯誤 |
| `Exception` | 🌊 海綿 | 所有 Python 常規錯誤 |

In [ ]:
class ShopSystemError(Exception): pass
class OutOfStockError(ShopSystemError): pass
class UserAuthError(ShopSystemError): pass
class PaymentError(ShopSystemError): pass
class CreditCardExpiredError(PaymentError): pass
class InsufficientBalanceError(PaymentError): pass

def process_payment(amount, balance, card_expired=False):
    if card_expired:
        raise CreditCardExpiredError("信用卡已過期，請更新卡片資訊")
    if amount > balance:
        raise InsufficientBalanceError(f"餘額不足：需要 {amount}，只有 {balance}")
    return f"付款 {amount} 元成功"

for test in [
    {"amount": 500,  "balance": 1000, "card_expired": False},
    {"amount": 5000, "balance": 1000, "card_expired": False},
    {"amount": 500,  "balance": 1000, "card_expired": True},
]:
    try:
        print(f"🟢 {process_payment(**test)}")
    except CreditCardExpiredError as e:
        print(f"🔴 [卡片問題] {e}")
    except InsufficientBalanceError as e:
        print(f"🟡 [餘額問題] {e}")
    except PaymentError as e:
        print(f"🟠 [其他付款問題] {e}")
    except ShopSystemError as e:
        print(f"💥 [系統問題] {e}")

> ❌ **新手常犯：父類別放前面，子類別永遠不會被執行到**
> ```python
> except PaymentError:           # 大網先下
>     print("付款問題")
> except InsufficientBalanceError:  # 💀 永遠到不了這裡
>     print("餘額不足")
> ```

> 💡 **多個錯誤做同樣處理？用 Tuple 合併：**
> ```python
> except (InsufficientBalanceError, CreditCardExpiredError) as e:
>     return_to_home_page()
> ```

---
## 6. 📁 實務專案結構：`exceptions.py`

```
shop_system/
├── main.py
├── payment.py
└── exceptions.py    ← 所有自定義例外集中在這裡
```

In [ ]:
# === exceptions.py ===
class ShopSystemError(Exception): pass
class PaymentError(ShopSystemError): pass
class InsufficientBalanceError(PaymentError): pass

# === payment.py ===
def process_payment(amount, balance):
    if amount > balance:
        raise InsufficientBalanceError(f"餘額不足：需要 {amount}，只有 {balance}")
    return f"付款 {amount} 元成功"

# === main.py ===
try:
    print(process_payment(5000, 1000))
except ShopSystemError as e:
    print(f"購物失敗：{e}")
    print(f"實際例外型態：{type(e).__name__}")

---
## 7. 🏦 整合範例：銀行轉帳系統

| 修改參數 | 觸發例外 | 被哪個 except 接住 |
|---|---|---|
| `amount="五十"` | `TypeError` | `except TypeError` |
| `to_account="123"` | `InvalidAccountError` | `except BankError`（父類別） |
| `amount=5000` | `InsufficientFundsError` | `except InsufficientFundsError` |
| `amount=500` | 無例外 | `else` 區塊執行 |

In [ ]:
class BankError(Exception): pass
class InsufficientFundsError(BankError): pass
class InvalidAccountError(BankError): pass

def transfer_money(from_account, to_account, amount, balance):
    if not isinstance(amount, (int, float)):
        raise TypeError("轉帳金額必須是數字！")
    if len(to_account) != 5:
        raise InvalidAccountError(f"帳號 {to_account} 格式錯誤，長度需為 5 碼。")
    if amount > balance:
        raise InsufficientFundsError(f"轉帳 {amount} 元失敗，餘額僅 {balance} 元。")
    return f"成功轉帳 {amount} 元至 {to_account}，剩餘 {balance - amount} 元。"

def run_atm_system():
    my_balance = 1000
    try:
        result = transfer_money("A0001", "B0002", 5000, my_balance)  # ← 修改這行
    except TypeError as e:
        print("🔴 【輸入格式錯誤】", e)
    except InsufficientFundsError as e:
        print("🟡 【交易拒絕】", e)
    except BankError as e:
        print("🟠 【銀行業務異常】", e)
    except Exception as e:
        print("💥 【未知嚴重錯誤】", e)
        raise
    else:
        print("🟢", result)
    finally:
        print("--- 交易流程結束 ---")

run_atm_system()

---
## 8. 自定義例外帶附加屬性

In [ ]:
class ValidationError(Exception):
    def __init__(self, field, message):
        self.field = field
        super().__init__(f"[{field}] {message}")

def validate_user(name, age):
    if not name:
        raise ValidationError(field="name", message="姓名不能為空")
    if not isinstance(age, int) or age < 0:
        raise ValidationError(field="age", message="年齡必須是正整數")

try:
    validate_user("", 25)
except ValidationError as e:
    print(f"驗證失敗：{e}")
    print(f"出錯欄位：{e.field}")

---
## 9. `raise from None` — 切斷例外鏈

In [ ]:
class ConfigError(Exception):
    pass

config = {"host": "localhost"}

print("=== 保留例外鏈 ===")
try:
    try:
        port = config["port"]
    except KeyError as e:
        raise ConfigError("設定檔缺少 port") from e
except ConfigError as e:
    print(f"錯誤：{e}，原因：{e.__cause__}")

print()
print("=== 切斷例外鏈（from None）===")
try:
    try:
        port = config["port"]
    except KeyError:
        raise ConfigError("設定檔缺少 port") from None
except ConfigError as e:
    print(f"錯誤：{e}，原因：{e.__cause__}")

---
## 10. 表單多欄位驗證

In [ ]:
class FormValidationError(Exception):
    def __init__(self, errors: list):
        self.errors = errors
        super().__init__(f"表單驗證失敗，共 {len(errors)} 個錯誤")

def validate_form(name, email, age):
    errors = []
    if not name:
        errors.append("姓名不能為空")
    if "@" not in email:
        errors.append("email 格式不正確")
    if not isinstance(age, int) or age < 18:
        errors.append("年齡必須是 18 歲以上的整數")
    if errors:
        raise FormValidationError(errors)

try:
    validate_form("", "not-an-email", 15)
except FormValidationError as e:
    print(f"❌ {e}")
    for i, err in enumerate(e.errors, 1):
        print(f"   {i}. {err}")

---
## 11. 重試機制 (Retry Pattern)

In [ ]:
import random

def unstable_connection():
    if random.random() < 0.7:
        raise ConnectionError("連線失敗")
    return "連線成功！"

def fetch_with_retry(max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            result = unstable_connection()
            print(f"第 {attempt} 次嘗試：{result}")
            return result
        except ConnectionError as e:
            print(f"第 {attempt} 次嘗試失敗：{e}")
            if attempt == max_retries:
                raise
            print("   → 重試中...")

try:
    fetch_with_retry(max_retries=3)
except ConnectionError:
    print("\n❌ 重試 3 次後仍無法連線。")

---
## 12. 狀態機 (State Machine) — 非法狀態轉換

In [ ]:
class InvalidStateError(Exception):
    def __init__(self, current, next_state):
        super().__init__(f"不能從 [{current}] 轉換到 [{next_state}]")

VALID_TRANSITIONS = {
    "待付款": ["已付款", "已取消"],
    "已付款": ["已出貨"],
    "已出貨": ["已送達"],
    "已送達": [],
    "已取消": [],
}

class Order:
    def __init__(self, order_id):
        self.order_id = order_id
        self.state = "待付款"

    def transition(self, next_state):
        if next_state not in VALID_TRANSITIONS.get(self.state, []):
            raise InvalidStateError(self.state, next_state)
        print(f"訂單 {self.order_id}：{self.state} → {next_state}")
        self.state = next_state

order = Order("ORD-001")
order.transition("已付款")
order.transition("已出貨")

try:
    order.transition("已取消")
except InvalidStateError as e:
    print(f"\n❌ {e}")

---
## 13. 自訂例外訊息格式 — `__str__` / `__repr__`

讓例外印出時的訊息更清楚、更豐富，方便 Debug。

| 方法 | 用途 |
|---|---|
| `__str__` | `print(e)` 或 `str(e)` 時顯示的訊息 |
| `__repr__` | 開發工具、log 系統用來表示物件的格式 |

In [ ]:
class InsufficientFundsError(Exception):
    def __init__(self, needed, available):
        self.needed = needed
        self.available = available
        self.shortage = needed - available

    def __str__(self):
        # print(e) 時顯示的訊息 — 給使用者看的
        return f"餘額不足 {self.shortage} 元（需要 {self.needed}，只有 {self.available}）"

    def __repr__(self):
        # repr(e) — 給開發者 / log 看的，顯示完整物件資訊
        return (f"InsufficientFundsError("
                f"needed={self.needed}, available={self.available})")

try:
    raise InsufficientFundsError(needed=5000, available=1000)
except InsufficientFundsError as e:
    print(f"str  → {e}")          # 使用者友善訊息
    print(f"repr → {repr(e)}")    # 開發者除錯資訊
    print(f"缺少：{e.shortage} 元")  # 透過屬性拿到數字

---
## 14. Context Manager + raise — 資源自動清理

搭配 `with` 語法，確保不管有沒有例外，資源（DB 連線、檔案、鎖）都會被正確釋放。

```
with db_transaction():
    ...執行業務邏輯...
    ↓ 若發生例外
    自動 rollback → 例外繼續往上傳
```

In [ ]:
from contextlib import contextmanager

@contextmanager
def db_transaction():
    """模擬資料庫交易的 Context Manager"""
    print("[DB] 開始交易 (BEGIN)")
    try:
        yield           # 把控制權交給 with 區塊
        print("[DB] 提交交易 (COMMIT)")
    except Exception:
        print("[DB] 回滾交易 (ROLLBACK)")
        raise           # 把例外繼續往上傳，不吃掉它
    finally:
        print("[DB] 關閉連線")


# 情境 1：正常執行
print("=== 正常情境 ===")
with db_transaction():
    print("  執行 SQL: INSERT INTO orders ...")

print()

# 情境 2：中途發生例外
print("=== 例外情境 ===")
try:
    with db_transaction():
        print("  執行 SQL: UPDATE balance ...")
        raise ValueError("金額不合法！")   # 模擬業務邏輯錯誤
        print("  這行不會執行")
except ValueError as e:
    print(f"[上層] 捕捉到：{e}")

---
## 15. Warning vs raise — 什麼時候不該讓程式 Crash？

| 情況 | 使用 | 說明 |
|---|---|---|
| 狀態嚴重不對 | `raise` | 程式必須停止，不能繼續 |
| 可繼續但需提醒 | `warnings.warn()` | 程式繼續執行，但留下警告訊息 |

常見的 Warning 類別：
- `DeprecationWarning` — 功能即將廢棄
- `UserWarning` — 一般使用者警告
- `RuntimeWarning` — 執行時期的潛在問題

In [ ]:
import warnings

# 1. DeprecationWarning — 函式即將廢棄，但仍可用
def old_calculate(x):
    warnings.warn(
        "old_calculate() 已棄用，請改用 new_calculate()",
        DeprecationWarning,
        stacklevel=2   # 讓警告指向呼叫者，而不是這個函式內部
    )
    return x * 2

# 2. UserWarning — 輸入有點奇怪但不至於錯
def set_discount(rate):
    if rate > 0.5:
        warnings.warn(f"折扣 {rate:.0%} 超過 50%，請確認是否正確", UserWarning)
    return rate

# 讓 warnings 顯示出來（預設某些類型會被過濾）
warnings.simplefilter("always")

print("=== DeprecationWarning ===")
result = old_calculate(10)   # 程式照常執行，但會印出警告
print(f"結果：{result}")

print()
print("=== UserWarning ===")
rate = set_discount(0.8)     # 程式繼續，不會 crash
print(f"折扣率：{rate:.0%}")

print()
print("=== 對比：raise 讓程式停止 ===")
def set_discount_strict(rate):
    if rate > 1.0:
        raise ValueError(f"折扣不能超過 100%")  # 這才是真的不允許
    return rate

try:
    set_discount_strict(1.5)
except ValueError as e:
    print(f"❌ {e}")

---
## 練習：自己設計一個例外系統

試著為「線上購物系統」設計 3 層例外類別並實作：

```
ShopError (基礎)
├── OutOfStockError        ← 商品缺貨
└── OrderError             ← 訂單相關（中間層）
    ├── InvalidCouponError ← 無效優惠券
    └── ShippingError      ← 配送問題
```

**進階挑戰：** 讓 `InvalidCouponError` 帶有 `coupon_code` 屬性，並自訂 `__str__`。

In [ ]:
# 在這裡實作你的購物系統例外
pass